In [16]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

from backend.extract_filter import classify_and_extract
from backend.data_utils import fetch_papers, verify_results, chunk_papers, format_query, clear
from backend.Database import create_index, save_embeddings_to_index, delete_index
from backend.retrieval import answer_query, clear_memory
from backend.pdf_ingest import ingest_uploaded_pdf
from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)


from time import sleep

In [53]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# from pinecone import Pinecone

indexes = pc.list_indexes()

# # delete all indexes
for idx in indexes:
     index_name = idx["name"]
     print(f" index: {index_name}")

 index: htmvn5001g


In [50]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# from pinecone import Pinecone

indexes = pc.list_indexes()

# # delete all indexes
for idx in indexes:
     index_name = idx["name"]
     pc.delete_index(index_name)
     print(f"Deleted index: {index_name}")



Deleted index: udojy9e9k8
Deleted index: dwgosrdom8
Deleted index: l82drlt8y5
Deleted index: kkg73somru


In [3]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

Index 'ou7oa7nxqn' created successfully!


In [ ]:
clear_memory()
clear()

def handleUserQuery(user_input, user_index):
    # for testing pdf uploading function
    # path = "Hacohen2022.pdf"
    # ingest_uploaded_pdf(file_path=path, user_index=index_name)

    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)
    print(classification)
 
    filters = classification['filters']
    print("\nRunning Search with filters:", filters)

    # if no academic query present, call llm without fetching paper
    if not format_query(filters):   
        question = classification['question']
        answer = answer_query(classification, index_obj, top_k=100)
        print("Generated Answer:")
        print(answer)
        return

    # Step 1: Fetch papers
    papers = fetch_papers(filters)

    # Step 2: Verify & Filter papers
    verified_papers = verify_results(papers, filters)
    print(f"[integration] Found {len(verified_papers)} verified paper(s).")
    
    # Determine number of papers to return; default to 1 if not provided
    try:
        num_papers = int(filters.get("max_results", 1))
    except ValueError:
        num_papers = 1
    num_papers = min(num_papers, len(verified_papers))

    # Step 3: Chunk papers
    chunks = chunk_papers(verified_papers[:num_papers])    # Get full paper content as chunks
    texts = [chunk["text"] for chunk in chunks]            # Extract just the text
    print("text:")
    print(texts)

    titles = [chunk["title"] for chunk in chunks]    # Extract the titles
    print('titles: ')
    print(titles)

    authors = [[author.lower() for author in chunk["author"]] for chunk in chunks]
    print("author:")
    print(authors)

    dates = [chunk["date"] for chunk in chunks]    # Extract the titles
    print('date: ')
    print(dates)


    if texts:
        # Save the full paper text chunks to the index in batches
        batch_size = 200
        for i in range(0, len(texts), batch_size): 
            print('i: ', i)
            text_batch = texts[i:i+batch_size]
            title_batch = titles[i:i+batch_size]     # 修改此处
            author_batch = authors[i:i+batch_size]    # 修改此处
            date_batch = dates[i:i+batch_size]        # 修改此处
            # 如果 sources 也是列表，也需做切片，否则传入列表或保持一致
            source_batch = ["arxiv"] * len(text_batch)
            save_embeddings_to_index(
                index_name=user_index, 
                text=text_batch, 
                titles=title_batch, 
                authors=author_batch, 
                dates=date_batch, 
                sources=source_batch,
                namespace="default"
            )
            sleep(1)



    #answer search question
    answer = answer_query(classification=classification, index_obj=index_obj, top_k=100)
    print("Generated Answer:")
    print(answer)
 
# Use a loop to enable continuous conversation
while True: 
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        # delete_index(index_name=index_name)
        clear_memory()
        clear()
        break
    handleUserQuery(user_query, index_name)


Memory cleared.
Loading uploaded PDF: Hacohen2022.pdf
Chunking uploaded document...
Storing 206 chunks to index: ou7oa7nxqn
pinecone gonna store: 
{'id': 'ee692ef1-919d-4a3f-9f68-fe339cd76f49', 'text': "'\n from user's uploaded PDF file: Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets\nGuy Hacohen 1 2 * Avihu Dekel1 * Daphna Weinshall1\nAbstract\nInvestigating active learning, we focus on the re-\nlation between the number of labeled examples\n(budget size), and suitable querying strategies.\nOur theoretical analysis shows a behavior rem-\niniscent of phase transition: typical examples are\nbest queried when the budget is low, while un-\nrepresentative examples are best queried when'\ntext from Hacohen2022.pdf\nauthored by ", 'title': 'hacohen2022.pdf', 'author': [], 'date': 0, 'source': 'uploaded_pdf'}
{'id': '140c95ea-3f52-4c36-b8db-e59d462a1f2c', 'text': "'\n from user's uploaded PDF file: best queried when the budget is low, while un-\nrepresentative exam

Token indices sequence length is longer than the specified maximum sequence length for this model (19507 > 1024). Running this sequence through the model will result in indexing errors
c:\Users\guoji\Group-3\backend\retrieval.py:219: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty i

After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retr

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Retrieved chunks: 
[{'text': "'\n from user's uploaded PDF file: Madison, 2009.\nShah, H., Tamuly, K., Raghunathan, A., Jain, P., and Netra-\npalli, P. The pitfalls of simplicity bias in neural networks.\nAdvances in Neural Information Processing Systems, 33:\n9573–9585, 2020.\nShui, C., Zhou, F., Gagné, C., and Wang, B. Deep active\nlearning: Uniﬁed and principled method for query and\ntraining. In International Conference on Artiﬁcial Intelli-\ngence and Statistics, pp. 1308–1318. PMLR, 2020.\nSiméoni, O., Budnik, M., Avrithis, Y ., and Gravier, G. Re-'\ntext from Hacohen2022.pdf\nauthored by ", 'title': 'hacohen2022.pdf', 'author': [], 'date': 0.0, 'source': 'uploaded_pdf'}, {'text': "'\n from user's uploaded PDF file: sarial active learning. In Proceedings of the IEEE/CVF\nInternational Conference on Computer Vision, pp. 5972–\n5981, 2019.\nTropp, J. A. An introduction to matrix concentration in-\nequalities. Found. Trends Mach. Learn., 8(1-2):1–230,\n2015. doi: 10.1561/2200000048.

...

Filter condition: 
{}
Retrieved chunks: 
[{'text': "'\n from user's uploaded PDF file: Active Learning (AL) aims to alleviate this problem (see\n*Equal contribution 1School of Computer Science and En-\ngineering, The Hebrew University of Jerusalem, Jerusalem, Is-\nrael 2Edmond & Lily Safra Center for Brain Sciences, The\nHebrew University of Jerusalem, Jerusalem, Israel. Corre-\nspondence to: Guy Hacohen <guy.hacohen@mail.huji.ac.il>,\nAvihu Dekel <avihu.dekel@mail.huji.ac.il>, Daphna Weinshall\n<daphna@cs.huji.ac.il>.\nProceedings of the 39 th International Conference on Machine'\ntext from Hacohen2022.pdf\nauthored by ", 'title': 'hacohen2022.pdf', 'author': [], 'date': 0.0, 'source': 'uploaded_pdf'}, {'text': "'\n from user's uploaded PDF file: Active Learning (AL) aims to alleviate this problem (see\n*Equal contribution 1School of Computer Science and En-\ngineering, The Hebrew University of Jerusalem, Jerusalem, Is-\nrael 2Edmond & Lily Safra Center for Brain Sciences, The\nHebrew

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Retrieved chunks: 
[{'text': "'\n from user's uploaded PDF file: of typical examples, which are likely to be representative\nof the entire dataset. To this end, TypiClust employs self-\nsupervised representation learning and then estimates each\npoint’s density in this representation. Diversity is obtained\nby clustering the dataset and sampling the densest point\nfrom each cluster (see Fig. 2).\nIn Section 4, we compare TypiClust to various AL strategies\nin the low-budget regime. TypiClust consistently improves'\ntext from Hacohen2022.pdf\nauthored by ", 'title': 'hacohen2022.pdf', 'author': [], 'date': 0.0, 'source': 'uploaded_pdf'}, {'text': "'\n from user's uploaded PDF file: of typical examples, which are likely to be representative\nof the entire dataset. To this end, TypiClust employs self-\nsupervised representation learning and then estimates each\npoint’s density in this representation. Diversity is obtained\nby clustering the dataset and sampling the densest point\nfrom eac

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Generated Answer:
content='The release date of the uploaded file "hacohen2022.pdf" is not specified in the provided information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 20647, 'total_tokens': 20670, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_b376dfbbd5', 'finish_reason': 'stop', 'logprobs': None} id='run-b5bd9687-c37e-422c-b473-859477db5943-0'
Memory cleared.


In [ ]:
# 